<a href="https://colab.research.google.com/github/Shemuel-bot/AI-ball-notebooks/blob/main/Ball_Trap_Brain_constructive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ball Trap Brain: constructive training (write the answer into the brain)

No search, no GA, no gradients. The routing (`alphabet_directions`, `alphabet_start_positions`, every deflector's `dest`) is left exactly as initialised. For each `(prompt, answer)` case we simulate where every ball goes and turn deflectors on its path into **holes** holding the answer symbols (or lock them as plain bouncers so they never emit by accident).

Cases use the format `cases = [(prompt, answer), ...]`.

Limits (worth knowing):
- It is a lookup table: it memorises the training cases and does not generalise.
- Capacity is bounded by the number of distinct ball start nodes, roughly `len(alphabet) * (2*direction_range + 1)`. Raise `direction_range` and the deflector count if cases fail.
- Two cases that send the same token to the same start node but need different symbols there conflict. The trainer backtracks over how the answer is split between balls and over which path nodes become holes, and reports what it could not satisfy.

In [ ]:
import random, copy, sys



def TokenizeInput(input, alphabet):
    input_str = str(input)
    matches = []
    for order, item in enumerate(alphabet):
        item_str = str(item)
        if not item_str:
            continue
        start_pos = 0
        while True:
            index = input_str.find(item_str, start_pos)
            if index == -1:
                break
            matches.append((index, -len(item_str), order, item))
            start_pos = index + 1
    matches.sort(key=lambda match: (match[0], match[1], match[2]))
    non_overlapping_items = []
    last_end_index = 0
    for match in matches:
        start_index = match[0]
        length = -match[1]
        item = match[3]
        if start_index >= last_end_index:
            non_overlapping_items.append(item)
            last_end_index = start_index + length
    return non_overlapping_items


class Ball_Trap_Brain:
    def __init__(self, deflectors, bounce, alphabet, output, holes_multiplier=1, direction_range=10):
        self.output = output
        self.holes_multiplier = holes_multiplier
        self.ball_energy = bounce * 2
        self.alphabet = alphabet
        self.direction_range = direction_range
        self.alphabet_directions = self.choose_alphabet_direction(len(alphabet))
        self.alphabet_start_positions = self.start_positions(len(alphabet), deflectors)
        self.deflectors_list = [[random.randint(0, deflectors - 1), bounce] for i in range(deflectors)]
        self.set_holes(output, deflectors, self.holes_multiplier)

    def set_holes(self, output, deflectors, holes_multiplier):
        for i in range(len(output)):
            for j in range(holes_multiplier):
                self.deflectors_list[random.randint(0, deflectors - 1)][1] = output[i]

    def choose_alphabet_direction(self, alphabet):
        return [random.randint(-self.direction_range, self.direction_range) for _ in range(alphabet)]

    def start_positions(self, alphabet, deflectors):
        return [random.randint(0, deflectors - 1) for i in range(alphabet)]

    def compute_input(self, input):
        directions, positions, travel = [], [], []
        input_arr = TokenizeInput(input, self.alphabet)
        for i in input_arr:
            if i in self.alphabet:
                index = self.alphabet.index(i)
                directions.append(self.alphabet_directions[index])
                positions.append(self.alphabet_start_positions[index])
        for i in range(len(directions)):
            travel.append(self.travel(copy.copy(directions), positions[i], i))
        return self.ricochet(travel, 10, self.ball_energy)

    def ricochet(self, travel, cap, ball_energy):
        output = []
        dl = self.deflectors_list
        n = len(dl)
        for t in travel:
            new_idx = t % n
            cap_idx = 0
            be = ball_energy
            while be > 0 and cap_idx < cap:
                dest, val = dl[new_idx]
                distance = abs(dest - new_idx)
                if not isinstance(val, int):
                    output.append(val)
                    be += 1
                    be -= distance
                    new_idx = dest
                    cap_idx += 1
                    continue
                be += val
                be -= distance
                new_idx = dest
                cap_idx += 1
        return output

    def compute_deflectors(self, input):
        directions, positions, travel = [], [], []
        input_arr = TokenizeInput(input, self.alphabet)
        for i in input_arr:
            if i in self.alphabet:
                index = self.alphabet.index(i)
                directions.append(self.alphabet_directions[index])
                positions.append(self.alphabet_start_positions[index])
        for i in range(len(directions)):
            travel.append(self.travel(copy.copy(directions), positions[i], i))
        return self.ricochet_deflectors(travel, 10, self.ball_energy)

    def ricochet_deflectors(self, travel, cap, ball_energy):
        deflectors_hit = []
        dl = self.deflectors_list
        n = len(dl)
        for t in travel:
            new_idx = t % n
            cap_idx = 0
            be = ball_energy
            while be > 0 and cap_idx < cap:
                dest, val = dl[new_idx]
                deflectors_hit.append(new_idx)
                distance = abs(dest - new_idx)
                if not isinstance(val, int):
                    be += 1
                    be -= distance
                    new_idx = dest
                    cap_idx += 1
                    continue
                be += val
                be -= distance
                new_idx = dest
                cap_idx += 1
        return deflectors_hit

    def travel(self, directions, position, index):
        # position-weighted, leave-one-out weighted average of the other tokens' directions
        total = 0
        weight_sum = 0
        for j, d in enumerate(directions):
            if j == index:
                continue
            weight = j + 1
            total += d * weight
            weight_sum += weight
        if weight_sum > 0:
            total = round(total / weight_sum)
        position += total
        return position

    def display_deflectors(self):
        for i, deflect in enumerate(self.deflectors_list):
            print(f"Deflector {i}: {deflect}")

## Constructive writer

In [ ]:
class OutOfBudget(Exception):
    pass


_FREE = object()   # node not decided yet
# state[i] is a symbol -> hole emitting that symbol, None -> plain bouncer, absent -> free


def ball_starts(brain, prompt):
    """Start deflector of every ball for `prompt` (mirrors compute_input)."""
    tokens = TokenizeInput(prompt, brain.alphabet)
    directions, positions = [], []
    for t in tokens:
        k = brain.alphabet.index(t)
        directions.append(brain.alphabet_directions[k])
        positions.append(brain.alphabet_start_positions[k])
    n = len(brain.deflectors_list)
    return [brain.travel(directions[:], positions[i], i) % n for i in range(len(directions))]


def answer_tokens(brain, answer):
    """Answer (str or list of symbols) -> list of output symbols."""
    if isinstance(answer, str):
        toks = TokenizeInput(answer, brain.output)
        if "".join(str(t) for t in toks) != answer:
            raise ValueError(f"answer {answer!r} has characters outside brain.output")
        return list(toks)
    return list(answer)


def predict(brain, prompt):
    return brain.compute_input(prompt)


def evaluate(brain, cases):
    """Exact-match accuracy of the brain on cases = [(prompt, answer), ...]."""
    if not cases:
        return 0.0
    ok = sum(brain.compute_input(p) == answer_tokens(brain, a) for p, a in cases)
    return ok / len(cases)


def write_cases(brain, cases, cap=10, rewrite_existing=True, max_nodes=200_000,
                shuffle=False, verbose=True):
    """
    Constructive 'training': decide, for every case, which deflectors on each ball's
    path become holes (and with which symbol) so the brain outputs `answer` for `prompt`.
    Nothing but deflector values (val) is changed; routing (dest, directions, start
    positions) is untouched.

    cases             : [(prompt, answer), ...]
    cap               : the cap used by brain.ricochet (10 in compute_input)
    rewrite_existing  : False -> holes already in the brain are kept as-is (locked)
    max_nodes         : search budget per case; case counts as failed if exceeded
    Returns a report dict. Cases that could not be satisfied are left unwritten.
    """
    dl = brain.deflectors_list
    energy0 = brain.ball_energy
    bounce = energy0 // 2
    if bounce <= 0:
        raise ValueError("bounce must be > 0")
    if any(isinstance(s, int) for s in brain.output):
        raise ValueError("output symbols must not be ints (ricochet treats ints as bounce values)")

    state = {}
    if not rewrite_existing:
        for i, (_, v) in enumerate(dl):
            if not isinstance(v, int):
                state[i] = v

    order = list(range(len(cases)))
    if shuffle:
        random.shuffle(order)

    failed = []

    def solve_case(prompt, answer):
        starts = ball_starts(brain, prompt)
        target = answer_tokens(brain, answer)
        k = len(starts)
        if k == 0:
            return len(target) == 0
        log = []
        budget = [max_nodes]
        sys.setrecursionlimit(max(sys.getrecursionlimit(), (k + 2) * (cap + 4) + 200))

        def plan(j, pos):
            # choose how many symbols ball j emits, then route it
            remaining = len(target) - pos
            if j == k:
                return remaining == 0
            if j == k - 1:
                counts = [remaining] if remaining <= cap else []
            else:
                ideal = -(-remaining // (k - j))
                counts = [ideal] + list(range(ideal - 1, -1, -1)) + list(range(ideal + 1, remaining + 1))
                counts = [c for c in counts if c <= min(remaining, cap)]
            for c in counts:
                if walk(j, pos, c, starts[j], energy0, 0, 0):
                    return True
            return False

        def walk(j, pos, c, idx, be, steps, e):
            budget[0] -= 1
            if budget[0] < 0:
                raise OutOfBudget
            if be <= 0 or steps >= cap:
                return e == c and plan(j + 1, pos + c)
            dest = dl[idx][0]
            dist = abs(dest - idx)
            s = state.get(idx, _FREE)
            if s is not _FREE:
                if s is None:                                   # locked plain
                    return walk(j, pos, c, dest, be + bounce - dist, steps + 1, e)
                if e < c and target[pos + e] == s:              # locked hole, right symbol
                    return walk(j, pos, c, dest, be + 1 - dist, steps + 1, e + 1)
                return False                                    # would emit a wrong/extra symbol
            if e < c:                                           # free: try making it a hole
                state[idx] = target[pos + e]
                log.append(idx)
                if walk(j, pos, c, dest, be + 1 - dist, steps + 1, e + 1):
                    return True
                del state[idx]
                log.pop()
            state[idx] = None                                   # free: make it plain
            log.append(idx)
            if walk(j, pos, c, dest, be + bounce - dist, steps + 1, e):
                return True
            del state[idx]
            log.pop()
            return False

        try:
            ok = plan(0, 0)
        except OutOfBudget:
            ok = False
        if not ok:
            for idx in reversed(log):
                state.pop(idx, None)
        return ok

    for ci in order:
        prompt, answer = cases[ci]
        if not solve_case(prompt, answer):
            failed.append(ci)

    # apply decisions to the brain
    for idx, s in state.items():
        dl[idx][1] = bounce if s is None else s

    ok_cases = [c for i, c in enumerate(cases) if i not in set(failed)]
    verified = evaluate(brain, ok_cases) if ok_cases else 0.0
    report = {
        "n_cases": len(cases),
        "n_written": len(ok_cases),
        "failed": sorted(failed),
        "deflectors_decided": len(state),
        "holes_written": sum(1 for s in state.values() if s is not None),
        "verified_on_written": verified,
    }
    if verbose:
        print(f"written {report['n_written']}/{report['n_cases']} cases | "
              f"holes {report['holes_written']} | decided deflectors {report['deflectors_decided']}/{len(dl)} | "
              f"re-verified exact match on written cases: {verified:.1%}")
    return report

## Demo: single-digit addition

In [ ]:
random.seed(1)

digits = list("0123456789")
alphabet = digits + ["+", "="]
output = digits

# Big direction_range + many deflectors = many distinct start nodes = more capacity
brain = Ball_Trap_Brain(deflectors=50000, bounce=10, alphabet=alphabet, output=output,
                        holes_multiplier=10, direction_range=10000)

cases = [(f"1+{b}=", str(1 + b)) for b in range(10000)]
random.shuffle(cases)
margin = int((len(cases)-1)*0.8)
train, test = cases[:margin], cases[margin:]

print("before:", evaluate(brain, train))
report = write_cases(brain, train)
print("after :", evaluate(brain, train))
print("failed case indices:", report["failed"])

before: 0.0
written 3326/7999 cases | holes 12309 | decided deflectors 20239/50000 | re-verified exact match on written cases: 100.0%
after : 0.4158019752469059
failed case indices: [50, 54, 67, 100, 139, 143, 194, 198, 204, 207, 208, 214, 223, 239, 240, 248, 264, 271, 285, 290, 320, 326, 328, 347, 359, 363, 369, 374, 383, 387, 403, 406, 437, 446, 449, 454, 458, 467, 475, 476, 491, 501, 502, 503, 507, 528, 531, 536, 537, 544, 558, 559, 560, 567, 573, 583, 587, 590, 596, 598, 601, 614, 616, 625, 628, 630, 645, 648, 654, 659, 661, 667, 670, 674, 677, 685, 696, 697, 700, 702, 708, 712, 714, 717, 721, 724, 726, 730, 731, 732, 733, 734, 737, 738, 739, 741, 744, 745, 749, 756, 757, 763, 764, 768, 769, 773, 775, 776, 777, 779, 780, 785, 790, 794, 805, 812, 817, 821, 826, 831, 832, 836, 840, 849, 853, 859, 861, 865, 866, 867, 868, 878, 881, 883, 889, 890, 894, 899, 905, 907, 909, 910, 912, 916, 921, 923, 924, 925, 941, 953, 962, 967, 970, 975, 979, 983, 995, 996, 997, 1002, 1003, 1006, 1010, 1

## Held-out cases (expect ~0: this is memorisation)

In [ ]:
print("held-out exact match:", evaluate(brain, test))
for p, a in test[:5]:
    print(p, "->", "".join(brain.compute_input(p)), "(expected", a + ")")

held-out exact match: 0.0
1+4912= -> 68 (expected 4913)
1+9697= -> 1 (expected 9698)
1+1999= -> 98 (expected 2000)
1+5608= -> 3 (expected 5609)
1+6112= ->  (expected 6113)


## Inspect how one training case is routed

In [ ]:
p, a = train[0]
print(p, "->", "".join(brain.compute_input(p)))
print("deflectors hit:", brain.compute_deflectors(p))
for i in sorted(set(brain.compute_deflectors(p))):
    print(i, brain.deflectors_list[i])

1+5944= -> 5945
deflectors hit: [5942, 42353, 10080, 40153, 14963, 17494, 31149]
5942 [4029, '5']
10080 [23283, '4']
14963 [29911, 10]
17494 [28182, 10]
31149 [808, 10]
40153 [46848, '5']
42353 [45987, '9']


## Options
- `rewrite_existing=False`: keep the holes created by `__init__` as locked, so only plain deflectors are rewritten.
- `shuffle=True`: random case order (rerun if the last few cases fail).
- `max_nodes`: per-case search budget. Raise it for long answers or large `bounce`, where paths are longer and the search is bigger.
- Output symbols must be non-int (strings), because `ricochet` treats ints as bounce values.